In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

archive_dir = Path("/Users/alechewitt/Desktop/Erdos_project/archive")
data_dir = Path("/Users/alechewitt/Desktop/Erdos_project/data")

# Optional: keep this smaller while testing
START_DATE = "2010-01-01"
END_DATE = "2022-12-31"
#START_DATE = "2020-02-19"
#END_DATE = "2020-03-23"

# Load all parquet files
files = sorted(archive_dir.glob("*.parquet"))

df = pd.concat(
    [pd.read_parquet(file) for file in files],
    ignore_index=True
)

# Clean column names: "[QUOTE_DATE]" -> "QUOTE_DATE"
df.columns = (
    df.columns
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
)

# Basic date parsing
df["QUOTE_DATE"] = pd.to_datetime(df["QUOTE_DATE"])
df["EXPIRE_DATE"] = pd.to_datetime(df["EXPIRE_DATE"])

# Optional date filter
df = df[(df["QUOTE_DATE"] >= START_DATE) & (df["QUOTE_DATE"] <= END_DATE)].copy()

# Convert important numeric columns
numeric_cols = [
    "UNDERLYING_LAST",
    "STRIKE",
    "C_BID",
    "C_ASK",
    "C_LAST",
    "C_IV",
    "C_VOLUME",
    "C_DELTA",
    "C_GAMMA",
    "C_VEGA",
    "C_THETA",
    "DTE",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# -----------------------------
# 1. Build real option data
# -----------------------------

options = pd.DataFrame({
    "date": df["QUOTE_DATE"],
    "expiration": df["EXPIRE_DATE"],
    "underlying_price": df["UNDERLYING_LAST"],
    "strike": df["STRIKE"],
    "bid": df["C_BID"],
    "ask": df["C_ASK"],
    "lastPrice": df["C_LAST"],
    "impliedVolatility": df["C_IV"],
    "volume": df["C_VOLUME"],
    "delta": df["C_DELTA"],
    "gamma": df["C_GAMMA"],
    "vega": df["C_VEGA"],
    "theta": df["C_THETA"],
})

# The shared loader filters to call options, so make this explicit.
options["option_type"] = "call"

# If your old project expects openInterest, add a placeholder
options["openInterest"] = 1

# Core calculated fields
options["mid_price"] = (options["bid"] + options["ask"]) / 2
options["T"] = (options["expiration"] - options["date"]).dt.days / 365
options["moneyness"] = options["strike"] / options["underlying_price"]
options["log_moneyness"] = np.log(options["moneyness"])

# Keep clean, liquid-ish call options
options = options.replace([np.inf, -np.inf], np.nan)
options = options.dropna(
    subset=[
        "date",
        "expiration",
        "underlying_price",
        "strike",
        "bid",
        "ask",
        "impliedVolatility",
        "mid_price",
        "T",
        "moneyness",
        "log_moneyness",
    ]
)

options = options[
    (options["bid"] > 0) &
    (options["ask"] > 0) &
    (options["impliedVolatility"] > 0) &
    (options["T"] > 7 / 365) &
    (options["T"] < 365 / 365) &
    (options["moneyness"] > 0.80) &
    (options["moneyness"] < 1.20)
].copy()

options = options.sort_values(["date", "expiration", "strike"])

# Save canonical real option datasets.
# historical_options.csv is what load_all_data() uses to avoid yfinance/synthetic fallback.
options.to_csv(data_dir / "historical_options.csv", index=False)
options.to_csv(data_dir / "backtest_options.csv", index=False)

# Save a surface dataset for the standalone Dupire notebook.
surface_options = options.copy()
surface_options.to_csv(data_dir / "surface_options.csv", index=False)

# -----------------------------
# 2. Build real SPY stock prices
# -----------------------------

stock = (
    df[["QUOTE_DATE", "UNDERLYING_LAST"]]
    .dropna()
    .rename(columns={
        "QUOTE_DATE": "date",
        "UNDERLYING_LAST": "close",
    })
)

stock = (
    stock.groupby("date", as_index=False)["close"]
    .median()
    .sort_values("date")
)

# Add simple compatibility columns
stock["open"] = stock["close"]
stock["high"] = stock["close"]
stock["low"] = stock["close"]
stock["volume"] = np.nan

stock.to_csv(data_dir / "spy_stock_prices.csv", index=False)

print("Saved:")
print(data_dir / "historical_options.csv", options.shape)
print(data_dir / "backtest_options.csv", options.shape)
print(data_dir / "surface_options.csv", surface_options.shape)
print(data_dir / "spy_stock_prices.csv", stock.shape)

Saved:
/Users/alechewitt/Desktop/Erdos_project/data/historical_options.csv (4073248, 19)
/Users/alechewitt/Desktop/Erdos_project/data/backtest_options.csv (4073248, 19)
/Users/alechewitt/Desktop/Erdos_project/data/surface_options.csv (4073248, 19)
/Users/alechewitt/Desktop/Erdos_project/data/spy_stock_prices.csv (3258, 6)
